# 04a — TGN-id: Degenerate Graph (single SENSOR node)

**Baseline for graph topology comparison against `04_tgn_training.ipynb`.**

## Graph structure (degenerate — original):
```
WELL-00 → SENSOR   (single shared destination)
WELL-01 → SENSOR
...
WELL-09 → SENSOR
edge_feat = [mean(all_sensors), delta_t]
```

The GRU memory of the single SENSOR node accumulates signals from all 10 well classes
→ information collapse → random-baseline AUC expected.

**Expected result:** AUC ≈ 0.50 — confirms graph topology matters more than model variant.

In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
import sys
sys.path.append('../../src')
from config import DATA_ROOT_3W, PROCESSED_DIR

torch.manual_seed(42)
np.random.seed(42)

PROCESSED_DIR = Path('../../data/processed')
MODELS_DIR    = Path('../../models')
MODELS_DIR.mkdir(exist_ok=True)

print(f'PyTorch: {torch.__version__}')
print(f'Device: {"GPU" if torch.cuda.is_available() else "CPU"}')

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


PyTorch: 2.9.1+cu128
Device: GPU


## 1. Load Data from Neo4j

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROCESSED_DIR = Path('/home/obiaggi/3w_processed')
MODELS_DIR    = Path('/home/obiaggi/TKG_Thesis/models')
MODELS_DIR.mkdir(exist_ok=True, parents=True)

SENSOR_COLS = ['P-PDG', 'P-TPT', 'T-TPT', 'P-MON-CKP', 'T-JUS-CKP', 'P-JUS-CKGL', 'QGL', 'T-PDG']
ROWS_PER_CLASS = 10000

CLASS_NAMES = {
    0:'Normal', 1:'BSW increase', 2:'DHSV closure', 3:'Severe slugging',
    4:'Flow instability', 5:'Productivity loss', 6:'Quick restriction',
    7:'Scaling', 8:'Hydrate', 9:'Other'
}

dfs = []
for type_num in range(10):
    f = PROCESSED_DIR / f'type_{type_num}.parquet'
    if not f.exists():
        print(f'  Missing type_{type_num} — skipping')
        continue
    d = pd.read_parquet(f)
    if 'event_type' in d.columns and 'class' not in d.columns:
        d = d.rename(columns={'event_type': 'class'})
    # DEGENERATE: one node per well type (not per instance)
    d['instance_id'] = f'WELL-{type_num:02d}'
    available = [c for c in SENSOR_COLS if c in d.columns]
    d = d.dropna(subset=available, how='all')
    if len(d) > ROWS_PER_CLASS:
        d = d.sample(ROWS_PER_CLASS, random_state=42)
    d['is_anomaly'] = (d['class'].fillna(0) > 0).astype(int)
    d['type_num'] = type_num
    # Aggregate all sensors to single value — information collapse
    d['value']     = d[available].fillna(0).mean(axis=1)
    d['sensor_id'] = 'SENSOR'   # single shared destination node
    dfs.append(d)
    print(f'  type_{type_num} ({CLASS_NAMES[type_num]:<20}): {len(d):>6,} rows | anomaly: {d["is_anomaly"].mean()*100:.0f}%')

df = pd.concat(dfs, ignore_index=True)

print(f'\nTotal rows  : {len(df):,}')
print(f'Normal      : {(df["is_anomaly"]==0).sum():,}')
print(f'Anomaly     : {(df["is_anomaly"]==1).sum():,}  ({df["is_anomaly"].mean()*100:.1f}%)')
print(f'Well nodes  : {df["instance_id"].nunique()} | Sensor nodes: 1 (SENSOR)')

## 2. Prepare TGN Data — Degenerate Graph

**Graph structure:**
```
WELL-00 → SENSOR
WELL-01 → SENSOR
...
WELL-09 → SENSOR
```

One edge per row. `edge_feat = [mean(sensors), delta_t]`.
All 10 well types share the same SENSOR destination node.
The SENSOR memory GRU accumulates all types simultaneously → no discriminative signal.

In [ ]:
from sklearn.model_selection import train_test_split as sk_split
from sklearn.preprocessing import StandardScaler
import torch


def prepare_tgn_data(df: pd.DataFrame):
    """Degenerate graph: WELL-XX → SENSOR (single shared node)."""
    wells   = df['instance_id'].unique().tolist()
    sensors = ['SENSOR']
    all_nodes = wells + sensors
    node2idx  = {n: i for i, n in enumerate(all_nodes)}
    num_nodes = len(all_nodes)

    scaler = StandardScaler()
    df = df.copy()
    df['value_norm'] = scaler.fit_transform(df[['value']].fillna(0))
    df['delta_t']    = df.groupby('instance_id').cumcount()
    df['delta_t']   /= df.groupby('instance_id')['delta_t'].transform('max').clip(lower=1)

    idx = np.arange(len(df))
    train_idx, test_idx = sk_split(
        idx, test_size=0.3, random_state=42, stratify=df['is_anomaly'].values)
    train_df = df.iloc[train_idx]
    test_df  = df.iloc[test_idx]

    def to_tensors(d):
        src = torch.tensor([node2idx[w] for w in d['instance_id']], dtype=torch.long)
        dst = torch.tensor([node2idx['SENSOR']] * len(d), dtype=torch.long)
        ef  = torch.tensor(d[['value_norm', 'delta_t']].values, dtype=torch.float32)
        dt  = torch.tensor(d['delta_t'].values, dtype=torch.float32).unsqueeze(1)
        y   = torch.tensor(d['is_anomaly'].values.astype(float), dtype=torch.float32)
        return src, dst, ef, dt, y

    print(f'Graph: {num_nodes} nodes ({len(wells)} well types + 1 SENSOR)')
    print(f'Train: {len(train_df):,} | Test: {len(test_df):,}')
    print(f'Train anomaly: {train_df["is_anomaly"].mean()*100:.1f}% | '
          f'Test anomaly: {test_df["is_anomaly"].mean()*100:.1f}%')

    return to_tensors(train_df), to_tensors(test_df), num_nodes


train_data, test_data, num_nodes = prepare_tgn_data(df)

## 3. Train TGN

In [ ]:
import torch, sys, numpy as np
from sklearn.metrics import classification_report, roc_auc_score, f1_score
sys.path.insert(0, '../../src')
try:
    from models.tgn import TGN
    print('TGN-id imported from src/models')
except ImportError as e:
    print(f'Import error: {e}')

if not df.empty and train_data is not None:
    model = TGN(num_nodes=num_nodes, memory_dim=64, message_dim=64, embed_dim=64, edge_dim=2)
    print(f'TGN parameters: {sum(p.numel() for p in model.parameters()):,}')
    print(f'Graph nodes   : {num_nodes} (degenerate — single SENSOR)')

    src, dst, ef, dt, y_tr = train_data
    n_normal  = float((y_tr == 0).sum())
    n_anomaly = float((y_tr == 1).sum())
    normal_weight = n_anomaly / max(n_normal, 1)
    print(f'Class weights — Normal: {normal_weight:.2f}x | Anomaly: 1.00x')

    import torch.nn as nn
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss(reduction='none')

    print('\nTraining TGN-id (5 epochs) — DEGENERATE graph...')
    for epoch in range(5):
        model.train()
        model.memory.reset()
        total_loss, n_batch = 0.0, 0
        for i in range(0, len(src), 512):
            s, d = src[i:i+512], dst[i:i+512]
            e, t_ = ef[i:i+512], dt[i:i+512]
            yb = y_tr[i:i+512]
            optimizer.zero_grad()
            # embed_src=False: classify on SENSOR memory (degenerate)
            pred = model(s, d, e, t_, embed_src=False)
            w = torch.where(yb == 0,
                            torch.full_like(yb, normal_weight),
                            torch.ones_like(yb))
            loss = (criterion(pred, yb) * w).mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            n_batch += 1
        print(f'  Epoch {epoch+1}/5 — loss: {total_loss/n_batch:.4f}')

    model.eval()
    src_t, dst_t, ef_t, dt_t, y_te = test_data
    all_scores = []
    with torch.no_grad():
        for i in range(0, len(src_t), 512):
            s = model(src_t[i:i+512], dst_t[i:i+512],
                      ef_t[i:i+512], dt_t[i:i+512],
                      update_memory=False, embed_src=False)
            all_scores.extend(s.numpy())

    scores = np.array(all_scores)
    y_np   = y_te.numpy()

    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.1, 0.9, 0.02):
        s = f1_score(y_np, (scores > t).astype(int), average='macro', zero_division=0)
        if s > best_f1:
            best_f1, best_t = s, t

    y_pred = (scores > best_t).astype(int)
    try:
        auc = roc_auc_score(y_np, scores)
    except Exception:
        auc = float('nan')

    print(f'\n[DEGENERATE GRAPH — TGN-id]')
    print(f'Best threshold: {best_t:.2f} (macro F1={best_f1:.3f})')
    print(classification_report(y_np, y_pred, target_names=['Normal', 'Anomaly'], digits=3))
    print(f'AUC-ROC: {auc:.4f}  ← expected ≈ 0.50 (random baseline)')

    torch.save({'model_state': model.state_dict(), 'num_nodes': num_nodes,
                'dataset': '3W', 'graph': 'degenerate'},
               MODELS_DIR / 'tgn_3w_degenerate.pt')
    print('Checkpoint saved.')
else:
    print('Load data first')